# 1.4 · LGD results

*1. Experiment 1 · notebook 1.4 of the story.* ← [1.3 · PD results](<1.3_pd_results.ipynb>) · [2.1 · PD fine-tuning](<../2. Experiment 2/2.1_pd_finetuning.ipynb>) →

**The benchmark — the experiment's answer.** Every trained Experiment 1 arm and
the shared reference models (the released TabICLv2, TabPFN, CatBoost, a linear model) scored by the
*same* code on the *same* splits and context cap, read from `output/results/lgd/eval/`. The
benchmark runs only once every arm of a track has trained, so until then every figure below is a
labelled placeholder — this notebook is safe to run at any point.

**Two questions, in the protocol's order** (`docs/EXPERIMENTAL_DESIGN.md` §5). *Which prior does the
development split select?* — the only data a prior may be chosen on. *And how does it do on the
holdout?* — untouched until the end, and what gets reported. Keeping the two apart is the
experiment's integrity: choosing on the holdout would invalidate it. The training-time view of the
same arms is [1.2 · LGD training](<1.2_lgd_training.ipynb>).

**How to read the metrics.** LGD is a bounded [0, 1] target with mass at both ends, so a point prediction from a bimodal predictive lands in the empty middle. The primary metrics are distributional — pinball, CRPS and boundary-mass calibration — with R² and RMSE alongside for comparability; CRPS scores the whole predictive, not its mean (`SYNTHESIS.md`; `papers/2026/02_Qu_TabICLv2` §I.7). Our nano-scale arms are expected to trail the frontier
reference in absolute terms; the claim is **prior contrast at matched compute**, not absolute
state of the art.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder
# (notebooks/1. Experiment 1/), from notebooks/, or from the root — then work FROM the root,
# so relative paths (config/...) resolve exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import results_plots, figures, style, literature

style.apply()   # ONE shared style: identical colours in every figure of every notebook
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "lgd"
EXP = "exp1"
# Reads output/manifests/ and output/results/ — whatever the run has written so far, so a PARTIAL
# sweep still renders. Constructing the saver clears THIS notebook's figure folder, and only it.
FIGS = figures.FigureSaver("1.4_lgd_results")

## The colour key

One vocabulary for every figure in every notebook, so a reader who has understood one figure can
read the next without its legend. Blue is our credit prior, grey the unmodified TabICL prior (the
control), orange real data, teal a value from `tfm-library`, amber one from outside it. Wherever a
figure varies the **credit fraction**, it runs from the control's grey (cf = 0) to our prior's blue
(cf = 1), monotone in lightness so the three levels still separate in a greyscale print.

In [ ]:
FIGS.save(style.show_palette(), "palette",
    caption="The shared colour vocabulary used on every axis of this notebook: each swatch names the prior, data source, literature overlay or annotation it marks.");

## A · Which prior does development select?

The selection, on development data only — equal weight per dataset, then per training seed; a configuration missing a development dataset is excluded, never rewarded for skipping a hard one (`src/eval/selection.py`).

### A1 · Configurations ranked on development

Every configuration on one axis, sorted, coloured by kind, with the training-seed spread as an error bar so nothing is ranked inside noise. The top of this ranking is the prior the next experiment inherits.

In [ ]:
FIGS.save(results_plots.overall_ranking(TASK, exp=EXP), "overall_ranking",
    caption="Development R\u00b2 per configuration, averaged over the development datasets and training seeds, as horizontal bars coloured by model kind with error bars for the training-seed standard deviation.");

## B · How does it do on the holdout?

The reported result: the same comparisons on the holdout datasets, which no selection decision has seen.

### B1 · Credit prior versus control

The figure the experiment exists for: credit arms against the control — TabICL's own prior by construction — and the external baselines, each point one model, each bar its group mean, on the holdout.

In [ ]:
FIGS.save(results_plots.credit_vs_control(TASK, exp=EXP, role="holdout"), "credit_vs_control",
    caption="Distribution of per-model mean R\u00b2 on the holdout datasets for credit-prior arms, control arms and external baselines; one point per model with a bar at each group mean.");

### B2 · Against the released reference

Each arm minus the released TabICLv2 on the holdout: anything right of zero beats a frontier model trained on orders of magnitude more compute. Trailing it is expected, and not the claim.

In [ ]:
FIGS.save(results_plots.beats_reference(TASK, exp=EXP, role="holdout"), "beats_reference",
    caption="Each trained arm's mean R\u00b2 on the holdout datasets minus the released TabICLv2's, one horizontal bar per arm, with a reference line at zero.");

### B3 · Every metric

The whole holdout scoreboard at once — a prior that helps ranking but hurts calibration splits across these panels.

In [ ]:
FIGS.save(results_plots.metric_grid(TASK, exp=EXP, role="holdout"), "metric_grid",
    caption="One panel per benchmark metric on the holdout datasets, each a bar per model kind (credit, control, baseline); the arrow in each title marks the improving direction.");

## C · Where does it hold?

No dataset hidden behind a mean: every dataset on both sides of the split, each labelled with its side.

### C1 · Every dataset

The best of each kind on each dataset, development first and then holdout — a prior that wins on average by helping one easy dataset is exposed here.

In [ ]:
FIGS.save(results_plots.per_dataset(TASK, exp=EXP), "per_dataset",
    caption="Best R\u00b2 per model kind on every real dataset as grouped bars, development datasets first and then holdout, each labelled with its side of the split.");

### C2 · The same, as one picture

The per-dataset scores as a heatmap, so the pattern across every dataset and kind is one glance rather than a wall of bars.

In [ ]:
FIGS.save(results_plots.per_dataset_heatmap(TASK, exp=EXP), "per_dataset_heatmap",
    caption="Best R\u00b2 of each model kind on each dataset as an annotated heatmap, development datasets first and then holdout on the vertical axis, model kinds on the horizontal axis.");

## D · Where the field sits

Our numbers beside the literature's — as context, never as a like-for-like target.

### D1 · There is no LGD landscape to draw

`tfm-library` contains **no credit-domain regression benchmark**: every published R², RMSE or CRPS in
it is on general regression suites (OpenML-CTR23, AMLB), and O'Prior's generator supports regression
but evaluates classification only (`papers/2026/05_Bouadi_ShapingThePrior`). That absence is the
point — bounded LGD targets are the literature's untested gap, and the gap this project works in. The
success criteria this project holds itself to (published LGD R² by model family) are its own, set in
`docs/EXPERIMENTAL_DESIGN.md` §5.1 from credit-risk literature outside the library; they are quoted
there, not drawn here, because no primary source for them is in the repository.

## Summary

The benchmark in text: what development selects (A), how it does on the holdout (B). Printed last,
in the order of the sections above, so `output/All_Results.md` reads the same story as this notebook
— followed by the `tfm-library` sources it leans on (pin `e5ce016`) and the figure inventory.

In [ ]:
print(results_plots.results_summary(TASK, exp=EXP))
print()
print(literature.references_md(["oprior_scope", "crps", "quantiles", "calibration_gap", "kumaraswamy"]))
print()
print(FIGS.summary())